In [ ]:
!pip install -q transformers datasets trl accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Load tokenizer and configure pad token
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Load model (Leave master weights in float32; TrainingArguments fp16 handles mixed precision)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = torch.cuda.is_available()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)

# 3. Format JSONL Q&A pairs into tokenized sequences
print("Loading and formatting dataset...")
raw_dataset = load_dataset("json", data_files="survival_data.jsonl", split="train")

def format_and_tokenize(batch):
    formatted_texts = []
    for inst, resp in zip(batch["instruction"], batch["response"]):
        messages = [
            {"role": "system", "content": "You are an emergency survival assistant. Provide direct, step-by-step, life-saving instructions without fluff."},
            {"role": "user", "content": inst},
            {"role": "assistant", "content": resp}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted_texts.append(text)
    
    tokenized = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=512,
        padding=False
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

dataset = raw_dataset.map(format_and_tokenize, batched=True, remove_columns=raw_dataset.column_names)

# Dynamic batch padding to resolve dimension mismatches
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./survival_model_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # Effective batch size = 8
    learning_rate=2e-5,
    num_train_epochs=3,
    warmup_steps=15,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=use_fp16,                    # PyTorch AMP handles mixed precision safely now
    report_to="none"
)

# 5. Trainer initialization
trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

# 6. Save the output
print("Saving fine-tuned model...")
trainer.save_model("./survival_model_final")
tokenizer.save_pretrained("./survival_model_final")
print("Saved successfully to ./survival_model_final")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "./survival_model_final"

print("Loading fine-tuned model for inference...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()

def generate_survival_response(instruction, system_prompt=None):
    if system_prompt is None:
        system_prompt = "You are an emergency survival assistant. Provide direct, step-by-step, life-saving instructions without fluff."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": instruction}
    ]
    
    # Format according to Qwen chat template
    text_input = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=350,
            temperature=0.3,       # Low temperature prevents hallucination in critical instructions
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Strip prompt tokens from output
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# Run Test Queries
# ==========================================
test_queries = (
    "I fell through thin river ice into freezing water. I managed to pull myself "
    "out onto the bank, but my clothes are soaked, the sun is setting, and the air "
    "temperature is below freezing. What are my exact, step-by-step priority actions right now?"
)

print(generate_survival_response(test_query))

for idx, query in enumerate(test_queries, 1):
    print(f"\n{'='*25} TEST QUERY {idx} {'='*25}")
    print(f"User: {query}\n")
    response = generate_survival_response(query)
    print("Model Response:\n" + response)

In [ ]:
test_query = (
    "What should I do if I discover my emergency kit is missing a critical item?"
)

print(generate_survival_response(test_query))